In [ ]:
# ==========================================
# Cell 1: 导入核心库
# ==========================================
import os
import numpy as np
import pandas as pd
import wfdb                     # 用于读取PhysioNet的电生理信号及注释文件
import neurokit2 as nk          # 生理信号处理库
import xgboost as xgb    
from pyrqa.time_series import TimeSeries
from pyrqa.settings import Settings
from pyrqa.analysis_type import Classic
from pyrqa.neighbourhood import FixedRadius
from pyrqa.metric import EuclideanMetric
from pyrqa.computation import RQAComputation
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm           # 进度条展示库
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['AR PL UKai CN'] # 解决制图时中文显示乱码问题

In [ ]:
# ==========================================
# Cell 2: 读取数据与预处理 (对应报告第4节 数据描述)
# ==========================================
# 提示：请确保 PAF Prediction Challenge 数据库的相对路径正确
data_path = './paf-prediction-challenge-database-1.0.0/'
sampling_frequency = 128        # 采样频率设置为128Hz

# 1. 提取远离房颤的SR信号 (NSR) 的 RR间期
nsr_RRI = []
for i in range(1, 51, 2):       # NSR患者的记录编号为奇数
    record_name = os.path.join(data_path, f'p{i:02d}')
    if os.path.exists(record_name + '.qrs'):
        # 读取心电图QRS波群的注释文件
        annotation = wfdb.rdann(record_name, 'qrs')
        # 通过计算相邻R波采样点的差值，除以采样率得到RR间期序列（单位：秒）
        rr_intervals = np.diff(annotation.sample) / sampling_frequency
        # 截取特定长度的片段用于特征分析
        for j in range(3):
            nsr_RRI.append(rr_intervals[31*j:31*(j+1)])

# 2. 提取接近房颤的 Pre-AF 信号 RR间期
preaf_RRI = []
for i in range(2, 51, 2):       # Pre-AF患者的记录编号为偶数
    record_name = os.path.join(data_path, f'p{i:02d}')
    if os.path.exists(record_name + '.qrs'):
        annotation = wfdb.rdann(record_name, 'qrs')
        rr_intervals = np.diff(annotation.sample) / sampling_frequency
        for j in range(3):
            # 取最接近房颤发作时刻（即序列末尾）的信号段
            start_index = len(rr_intervals) - 31 * (j + 1)
            end_index = len(rr_intervals) - 31 * j
            if start_index >= 0:
                preaf_RRI.append(rr_intervals[start_index:end_index])

# 3. 保存预处理好的 RR 间期数据，供后续提取特征使用
os.makedirs('./pro_data', exist_ok=True)
np.save('./pro_data/NSR_RR.npy', nsr_RRI)
np.save('./pro_data/PreAF_RR.npy', preaf_RRI)
print("数据提取与保存完成！")

数据提取与保存完成！


In [ ]:
# ==========================================
# Cell 3: 定量递归分析（RQA）特征提取函数
# ==========================================
def get_rqa_mea(file_path, threshold=0.05, measure='recurrence_rate'):
    """
    根据重构参数计算时间序列的 RQA 局部度量指标
    """
    rqa = []
    data = np.load(file_path, allow_pickle=True)
    for i in range(len(data)):
        # 构建相空间重构时间序列，嵌入维度 m=3，时间延迟 tau=2
        time_series = TimeSeries(data[i], embedding_dimension=3, time_delay=2)
        # 配置 RQA 参数：采用欧式距离及固定半径阈值(FixedRadius)
        settings = Settings(time_series,
                            analysis_type=Classic,
                            neighbourhood=FixedRadius(threshold),
                            similarity_measure=EuclideanMetric,
                            theiler_corrector=1) # Theiler 窗口用于去除自相关
        
        computation = RQAComputation.create(settings, verbose=False)
        result = computation.run()
        
        # 根据需求提取指定特征：递归度(RR)、确定度(DET)、熵(ENTR)、层状度(LAM)
        if measure == 'recurrence_rate':
            rqa.append(result.recurrence_rate)
        elif measure == 'determinism':
            rqa.append(result.determinism)
        elif measure == 'entropy_diagonal_lines':
            rqa.append(result.entropy_diagonal_lines)
        elif measure == 'laminarity':
            rqa.append(result.laminarity)
    return rqa

In [ ]:
# ==========================================
# Cell 4: XGBoost分类与最佳距离阈值寻优 (以确定度 DET 为例)
# ==========================================
# 设定阈值搜索空间，从 0.01 到 0.20，步长 0.01
thresholds = np.arange(0.01, 0.21, 0.01)
best_accuracy = 0
best_threshold = 0

print("正在进行阈值寻优，请稍候...")
for th in tqdm(thresholds):
    # 提取正负样本特征
    nsr_rqa = get_rqa_mea('./pro_data/NSR_RR.npy', threshold=th, measure='determinism')
    preaf_rqa = get_rqa_mea('./pro_data/PreAF_RR.npy', threshold=th, measure='determinism')
    
    # 组合数据集，确保特征矩阵维度正确 (n_samples, 1)
    X = np.vstack((np.array(nsr_rqa).reshape(-1, 1), np.array(preaf_rqa).reshape(-1, 1)))
    # NSR 标签为 0, PreAF 标签为 1
    y = np.hstack((np.zeros(len(nsr_rqa)), np.ones(len(preaf_rqa))))
    
    # 按 8:2 比例划分训练集与测试集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 实例化 XGBoost 分类器并进行训练
    clf = xgb.XGBClassifier(eval_metric='logloss')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    # 记录并更新最佳准确率及其对应的距离阈值
    accuracy = accuracy_score(y_test, y_pred)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = th

print(f"\n寻找完毕！当阈值为 {best_threshold:.2f} 时，取得最高准确率: {best_accuracy:.4f}")

正在进行阈值寻优，请稍候...


  0%|          | 0/20 [00:00<?, ?it/s]2 warnings generated.
/home/user/anaconda3/envs/syx_new/lib/python3.13/site-packages/pyopencl/__init__.py:519: CompilerWarning: Non-empty compiler output encountered. Set the environment variable PYOPENCL_COMPILER_OUTPUT=1 to see more.
  lambda: self._prg.build(options_bytes, devices),
100%|██████████| 20/20 [02:04<00:00,  6.25s/it]


寻找完毕！当阈值为 0.17 时，取得最高准确率: 0.7000


In [ ]:
# ==========================================
# Cell 5: 使用最佳参数打印最终的分类报告 (对应表3-1和表3-2)
# ==========================================
print(f"在最佳阈值 {best_threshold:.2f} 下的分类性能报告：")
# 使用寻优得到的最佳阈值重新提取特征
nsr_rqa_best = get_rqa_mea('./pro_data/NSR_RR.npy', threshold=best_threshold, measure='determinism')
preaf_rqa_best = get_rqa_mea('./pro_data/PreAF_RR.npy', threshold=best_threshold, measure='determinism')

X_best = np.vstack((np.array(nsr_rqa_best).reshape(-1, 1), np.array(preaf_rqa_best).reshape(-1, 1)))
y_best = np.hstack((np.zeros(len(nsr_rqa_best)), np.ones(len(preaf_rqa_best))))
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_best, y_best, test_size=0.2, random_state=42)

clf_best = xgb.XGBClassifier(eval_metric='logloss')
clf_best.fit(X_train_b, y_train_b)
y_pred_b = clf_best.predict(X_test_b)

# 打印包含精确率、灵敏度(recall, pos label)、特异度等指标的最终分类报告
print(classification_report(y_test_b, y_pred_b, target_names=['SR (0)', 'Pre-AF (1)']))

在最佳阈值 0.17 下的分类性能报告：
              precision    recall  f1-score   support

      SR (0)       0.67      0.88      0.76        16
  Pre-AF (1)       0.78      0.50      0.61        14

    accuracy                           0.70        30
   macro avg       0.72      0.69      0.68        30
weighted avg       0.72      0.70      0.69        30

